In [7]:
# import relevant libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS

from sklearn.mixture import GaussianMixture

import warnings
warnings.filterwarnings('ignore')

# Get Data

In [8]:
# get data
sheet_name_list = ['Period 1', 'Period 2', 'Period 3']
data_list = []

for sheet_name in sheet_name_list:
    data_period = pd.read_excel('regime_data.xlsx', sheet_name=sheet_name).iloc[1:].copy()
    data_period = data_period.set_index('Unnamed: 0')
    data_period.index.name = 'DATE'
    data_period.columns.name = 'ID'
    data_list.append(data_period)

data_df = pd.concat(data_list, axis=0)
data_df = data_df.sort_index()
data_df

ID,MXCXDMHR Index,LGY7TRUH Index,LUACTRUU Index,LF98TRUU Index,LP05TRUH Index,LP01TRUH Index,BCOMTR Index,M1EF Index,EMUSTRUU Index,MXEF0CX0 Index,USDJPY BGN Curncy,USDCHF BGN Curncy,AUDUSD BGN Curncy,NZDUSD BGN Curncy,USDCAD BGN Curncy,EURUSD BGN Curncy,GBPUSD BGN Curncy,LBUTTRUU Index,SPY US Equity
DATE,,,,,,,,,,,,,,,,,,,
1970-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1970-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1970-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1970-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1970-01-05,NaN,NaN,NaN,NaN,NaN,NaN,3.1177,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-27,713.99,846.9528,3555.1,2953.53,319.83,649.15,349.6279,894.06,1400.047,1874.5,159.42,0.7855,0.7186,0.5908,1.3627,1.1721,1.3535,378.3391,715.17
2026-04-28,710.84,846.1492,3552.86,2950.28,319.31,648.48,351.6852,886.9,1398.2716,1869.45,159.62,0.7893,0.7181,0.5885,1.3684,1.1712,1.3517,378.3211,711.69
2026-04-29,710.17,844.0966,3538.72,2945.34,318.6,648.07,357.2446,887.76,1396.1287,1866.59,160.41,0.7912,0.7116,0.5829,1.3685,1.1677,1.3475,377.1386,711.58


# Get Data

In [9]:
# get monthly data
data_df = data_df.resample("M").last().dropna()

data_df

ID,MXCXDMHR Index,LGY7TRUH Index,LUACTRUU Index,LF98TRUU Index,LP05TRUH Index,LP01TRUH Index,BCOMTR Index,M1EF Index,EMUSTRUU Index,MXEF0CX0 Index,USDJPY BGN Curncy,USDCHF BGN Curncy,AUDUSD BGN Curncy,NZDUSD BGN Curncy,USDCAD BGN Curncy,EURUSD BGN Curncy,GBPUSD BGN Curncy,LBUTTRUU Index,SPY US Equity
DATE,,,,,,,,,,,,,,,,,,,
2000-12-31,117.58,305.2734,1089.65,517.12,109.29,102.60,184.9170,100.00,265.0728,793.51,114.41,1.6111,0.5588,0.4437,1.4991,0.9427,1.4930,122.9687,131.1875
2001-01-31,121.39,310.0040,1120.84,555.86,110.58,111.79,180.5730,113.76,277.2077,799.84,116.57,1.6354,0.5506,0.4439,1.4978,0.9366,1.4646,125.5350,137.0200
2001-02-28,111.99,313.7589,1130.31,563.26,111.05,112.53,179.8120,104.84,273.7935,794.84,117.37,1.6681,0.5263,0.4305,1.5364,0.9236,1.4455,127.6530,123.9500
2001-03-31,106.47,316.7754,1137.24,550.00,112.02,107.15,172.1510,94.50,270.8515,784.12,126.34,1.7428,0.4856,0.4029,1.5757,0.8767,1.4161,128.8739,116.6900
2001-04-30,113.84,314.0998,1133.53,543.15,111.20,104.05,178.2080,99.15,270.0810,791.33,123.48,1.7330,0.5124,0.4138,1.5350,0.8891,1.4320,129.6006,124.9100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-31,685.98,848.1162,3551.52,2929.32,321.16,647.55,304.8839,834.38,1391.8607,1863.38,154.78,0.7730,0.6964,0.6021,1.3613,1.1851,1.3686,373.7130,691.9700
2026-02-28,696.02,862.3596,3597.34,2934.74,323.59,649.94,308.2337,880.23,1408.8088,1883.91,156.05,0.7693,0.7118,0.5998,1.3640,1.1812,1.3482,378.5965,685.9900
2026-03-31,653.35,843.7389,3526.16,2900.04,316.42,635.31,343.6952,765.25,1368.1591,1831.75,158.72,0.7995,0.6900,0.5747,1.3916,1.1553,1.3227,373.5345,650.3400
